# Reaction, Diffusion and Advection

Adding a reaction term to the heat equation creates **travelling fronts**: populations, genes or innovations spreading at a steady speed. Replacing diffusion by **advection**, transport at a velocity, raises a different numerical problem. The simplest stable schemes smear the solution, and centred ones make it oscillate.

In [ ]:
Pollis.packages("OrdinaryDiffEq", "Plots")

In [ ]:
using OrdinaryDiffEq, SparseArrays, LinearAlgebra, Plots

## Fisher-KPP: a front with a fixed speed

Fisher (1937) modelled an advantageous gene spreading through a population: $u_t = Du_{xx} + ru(1 - u)$. A region with $u = 1$ invades the empty region at the asymptotic speed $2\sqrt{rD}$. The speed approaches this value slowly, like $2 - 3/(2t)$ in these units. Here $D = r = 1$ on $[0, 100]$ with no-flux ends.

In [ ]:
L, n = 100.0, 400
h = L / n; xs = h .* ((1:n) .- 0.5)   # cell centres
A = spdiagm(-1 => ones(n - 1), 0 => fill(-2.0, n), 1 => ones(n - 1)); A[1, 1] = A[n, n] = -1.0; A = A ./ h^2
fisher(u, p, t) = A * u .+ u .* (1 .- u)
sol = solve(ODEProblem(fisher, [x < 5 ? 1.0 : 0.0 for x in xs], (0.0, 40.0)), FBDF(); reltol=1e-6, abstol=1e-9)
plot(xs, [sol(t) for t in 0:8:40], lw=2, label=permutedims(["t = $t" for t in 0:8:40]), xlabel="x", ylabel="u", title="Fisher-KPP: an invading front")

In [ ]:
function front(t)   # where u crosses 1/2, by linear interpolation
    u = sol(t); i = findfirst(<(0.5), u)
    return xs[i - 1] + (0.5 - u[i - 1]) / (u[i] - u[i - 1]) * h
end
ts = 10:5:40; pos = front.(ts)
[(from = ts[k], to = ts[k + 1], speed = (pos[k + 1] - pos[k]) / 5, asymptotic = 2.0, with_correction = 2 - 3 / (2 * (ts[k] + 2.5))) for k in 1:length(ts)-1]

## Advection: upwind and centred differences

$u_t + a u_x = 0$ moves any profile at speed $a$ without changing its shape. On a periodic domain of length 1 with $a = 1$, the profile returns to its start at $t = 1$. Two semi-discretisations:

- **Upwind:** $u_x \approx (u_i - u_{i-1})/h$. It is stable but adds numerical diffusion $\tfrac{ah}{2}u_{xx}$.
- **Centred:** $u_x \approx (u_{i+1} - u_{i-1})/(2h)$. It has no diffusion, but it is dispersive: short waves lag behind and leave ripples.

In [ ]:
m = 100; dx = 1 / m; xg = dx .* (0:m-1)
g = exp.(-(xg .- 0.5) .^ 2 ./ (2 * 0.05^2))
upwind(u, p, t) = -(u .- circshift(u, 1)) ./ dx
centred(u, p, t) = -(circshift(u, -1) .- circshift(u, 1)) ./ (2dx)
su = solve(ODEProblem(upwind, g, (0.0, 1.0)), Tsit5(); reltol=1e-8, abstol=1e-10)
sc = solve(ODEProblem(centred, g, (0.0, 1.0)), Tsit5(); reltol=1e-8, abstol=1e-10)
plot(xg, [g su.u[end] sc.u[end]], lw=2, label=["exact (one full turn)" "upwind" "centred"], xlabel="x", title="After one trip around the periodic domain")

Numerical diffusion widens the Gaussian: its variance grows by $a h t$. So the upwind peak falls to $\sqrt{0.05^2 / (0.05^2 + ah)} \approx 0.447$, which the simulation reproduces.

In [ ]:
(upwind_peak = maximum(su.u[end]), predicted_peak = sqrt(0.05^2 / (0.05^2 + dx)), centred_peak = maximum(sc.u[end]), centred_undershoot = minimum(sc.u[end]))

## Advection-diffusion

A pollutant in a river both drifts and spreads: $u_t + a u_x = D u_{xx}$. When physical diffusion $D$ is much larger than $ah/2$, upwind's numerical diffusion is harmless. The **cell Peclet number** $ah/(2D)$ measures this; below 1, even centred differences stay free of ripples.

In [ ]:
Dphys = 0.002
Ap = spdiagm(-1 => ones(m - 1), 0 => fill(-2.0, m), 1 => ones(m - 1)); Ap[1, m] = Ap[m, 1] = 1.0; Ap = Ap ./ dx^2
advdiff(u, p, t) = centred(u, p, t) .+ Dphys .* (Ap * u)
sa = solve(ODEProblem(advdiff, g, (0.0, 1.0)), Tsit5(); reltol=1e-8, abstol=1e-10)
exact_ad = sqrt(0.05^2 / (0.05^2 + 2Dphys)) .* exp.(-(xg .- 0.5) .^ 2 ./ (2 * (0.05^2 + 2Dphys)))
display(plot(xg, [exact_ad sa.u[end]], lw=2, label=["exact" "centred + diffusion"], xlabel="x"))
(cell_peclet = dx / (2Dphys), largest_error = maximum(abs.(sa.u[end] .- exact_ad)))

## Exercises

1. Change $D$ to 4 in Fisher-KPP. Does the front speed double, as $2\sqrt{rD}$ predicts?
2. Start Fisher-KPP from a small bump in the middle instead of a step. How many fronts form?
3. Refine the advection grid to $m = 400$. How much of the upwind smearing remains?
4. The panel's Heat Equation tab accepts a reaction term. Enter `u * (1 - u)` with $D = 0.001$ and a step initial profile, and measure the front speed with Probes.